# LotLogic YOLO Plate Detector — Charlotte Fine-tune

**Runtime → Change runtime type → T4 GPU** before running.

1. Drag `yolo-dataset.zip` into the Colab file panel (left sidebar).
2. Run all cells.
3. At the end, download `lotlogic-plate.onnx` from the file panel.
4. Drop the .onnx into the Railway sidecar and bounce.

In [ ]:
# Cell 1 — install ultralytics + onnx tooling
!pip install -q ultralytics onnx onnxruntime onnxslim

In [ ]:
# Cell 2 — unpack the dataset you uploaded
import zipfile, os
assert os.path.exists('/content/yolo-dataset.zip'), 'Upload yolo-dataset.zip to /content first'
with zipfile.ZipFile('/content/yolo-dataset.zip') as z:
    z.extractall('/content/')
# Patch dataset.yaml absolute path so ultralytics finds the images on Colab
yaml_path = '/content/yolo-dataset/dataset.yaml'
with open(yaml_path) as f:
    txt = f.read()
txt = txt.replace('path: /Users/gabe/lotlogic/yolo-dataset', 'path: /content/yolo-dataset')
with open(yaml_path, 'w') as f:
    f.write(txt)
print(open(yaml_path).read())
!ls /content/yolo-dataset/images/train | wc -l
!ls /content/yolo-dataset/images/val | wc -l

In [ ]:
# Cell 3 — train. yolov9t = tiny variant, ~3M params, ideal for ALPR sidecar.
# 100 epochs is overkill for 503 images but cheap on T4 (~30 min).
# imgsz=640 matches the sidecar's existing pipeline.
from ultralytics import YOLO
model = YOLO('yolov9t.pt')
results = model.train(
    data='/content/yolo-dataset/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    project='/content/lotlogic-finetune',
    name='train',
    patience=20,        # stop early if no val improvement
    cache='ram',        # tiny dataset, keep in memory
    plots=True,
)

In [ ]:
# Cell 4 — see the validation curves and confusion / PR plots
from IPython.display import Image, display
for fname in ['results.png', 'PR_curve.png', 'confusion_matrix.png', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    p = f'/content/lotlogic-finetune/train/{fname}'
    if os.path.exists(p):
        print(p)
        display(Image(p))

In [ ]:
# Cell 5 — export to ONNX, dynamic shape, opset 12 (compatible with onnxruntime 1.16+)
best = YOLO('/content/lotlogic-finetune/train/weights/best.pt')
onnx_path = best.export(format='onnx', imgsz=640, opset=12, dynamic=True, simplify=True)
print('Wrote', onnx_path)
import shutil
shutil.copy(onnx_path, '/content/lotlogic-plate.onnx')
print('Download from file panel: /content/lotlogic-plate.onnx')
!ls -lh /content/lotlogic-plate.onnx

In [ ]:
# Cell 6 — quick sanity check on a held-out val image
import glob, random
val_imgs = glob.glob('/content/yolo-dataset/images/val/*.jpg')
sample = random.sample(val_imgs, min(4, len(val_imgs)))
preds = best.predict(sample, conf=0.25, save=True, project='/content', name='predict')
for p in glob.glob('/content/predict/*.jpg'):
    display(Image(p))